# 微笑视频模拟调用样例

In [1]:
# 加载所需要的包
import os
import cv2
import glob
import base64
import time
import requests
import json
import urllib
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Video, display

<h2>定义调用规则</h2>
请根据您从我方获取的信息修改以下代码块

In [2]:
# 朝厚服务请求地址，随api文档发送
base_url = "<服务请求地址>"

# 朝厚文件服务地址，随api文档发送
file_server_url = "<服务文件服务器地址>"

# 必须传入鉴权 Header。请保护好TOKEN!!! 如果泄露请立即联系我们重置，所有使用该TOKEN的任务都会向您的账户计费
zh_token = "<贵司服务Token, 随合同发送>" # 调用所有的API都必须传入token用作鉴权

user_group = "APIClient" # 用户组，一般为 APIClient

# 贵司user_id, 随api文档发送
user_id = "<贵司user_id>"

# 如果您收到了creds.json, 下面将直接读取
if os.path.exists('../../creds.json'):
    creds = json.load(open('../../creds.json', 'r'))
    base_url = creds['base_url']
    file_server_url = creds['file_server_url']
    zh_token = creds['zh_token']
    user_id = creds['user_id']
    print("loaded creds from creds.json")

loaded creds from creds.json


### 使用文件方式上传

In [3]:
def upload_file(file_name):
    ext = file_name.split('.')[-1]
    data = open('../../data/' + file_name, 'rb').read()
    resp = requests.get(file_server_url + f"/scratch/{user_group}/{user_id}/upload_url?" +
                        f"postfix={ext}", # 必须指定 postfix, 即文件后缀名
                        headers={"X-ZH-TOKEN": zh_token}) # 获取带签名的上传地址
    resp.raise_for_status()

    upload_url = resp.text[1:-1] # 返回为一个单字符串JSON "string", 这里也可以用json.loads(resp.text)

    resp = requests.put(upload_url, data) # 上传至云储存服务不需要带鉴权头

    resp.raise_for_status()
    path = "/".join(urllib.parse.urlparse(upload_url).path.lstrip("/").split("/")[3:])
    urn = f"urn:zhfile:o:s:{user_group}:{user_id}:{path}"
    return urn

def run_job_and_get_results(json_call, timeout_sec):
    headers = {
      "Content-Type": "application/json",
      "X-ZH-TOKEN": zh_token
    }

    url = base_url + '/run'

    response = requests.request("POST", url, headers=headers, data=json.dumps(json_call))
    response.raise_for_status()
    create_result = response.json()
    run_id = create_result['run_id']
    print("workflow id is", run_id)
    url = base_url + f"/run/{run_id}"

    start_time = time.time()
    while time.time()-start_time < timeout_sec:
        time.sleep(0.3)
        response = requests.request("GET", url, headers=headers)
        result = response.json()
        if result['completed'] or result['failed']:
            break

    if not result['completed']:
        if result['failed']:
            raise ValueError("API failed due to " + str(result['reason_public']))
        raise TimeoutError("API timeout")

    print("API finished in {}s".format(time.time()-start_time))
    url = base_url + f"/data/{run_id}"
    response = requests.request("GET", url, headers=headers)
    return response.json()

def show_video(urn, output_path="result.mp4", show_info=True):
    """
    从文件服务器下载视频并保存到本地

    :param urn: API 返回的视频 URN
    :param output_path: 本地保存路径
    :param show_info: 是否打印视频信息
    """

    # 下载视频二进制
    resp = requests.get(
        file_server_url + "/file/download",
        params={"urn": urn},
        headers={"X-ZH-TOKEN": zh_token},
        timeout=60
    )
    resp.raise_for_status()

    # 保存为本地视频文件
    with open(output_path, "wb") as f:
        f.write(resp.content)

    print(f"Video saved to: {output_path}")

    if not show_info:
        return

    # 使用 OpenCV 读取视频信息
    cap = cv2.VideoCapture(output_path)
    if not cap.isOpened():
        print("Warning: video saved but cannot be opened by OpenCV")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration = frame_count / fps if fps > 0 else 0

    cap.release()

    print(
        f"Video info:\n"
        f"  Resolution : {width} x {height}\n"
        f"  FPS        : {fps:.2f}\n"
        f"  Frames     : {frame_count}\n"
        f"  Duration   : {duration:.2f} sec"
    )


## 矫正后微笑视频模拟

以下代码输出矫正后微笑模拟，具体可参考 https://www.chohotech.com/docs/cloud-zh/#/module/smile-video-simulation-1

In [4]:
json_call = {
  "spec_group": "smile",
  "spec_name": "smile-video-simulation",
  "spec_version": "1.0-snapshot",
  "user_group": user_group,
  "user_id": user_id,
  "input_data": {"video": upload_file("smile.mp4")}
}
result = run_job_and_get_results(json_call, 100)

workflow id is sa_service_1766733606-d9765d36-f69c-48bc-95c3-2ca4cdf5ccb4
API finished in 27.794500589370728s


In [5]:
print(result)

{'result': {'video': 'urn:zhfile:o:wfd:APIClient:ZH-api:wfdata/sa_service_1766733606-d9765d36-f69c-48bc-95c3-2ca4cdf5ccb4/output-files/result/video'}}


### 视频结果展示

In [8]:
show_video(result['result']['video'])

Video saved to: result.mp4
Video info:
  Resolution : 720 x 1280
  FPS        : 30.00
  Frames     : 153
  Duration   : 5.10 sec


In [9]:
display(Video("result.mp4", embed=True))